## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


In [ ]:
import os
os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY", "")

In [ ]:
!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login(key=os.environ["WANDB_API_KEY"]) if os.getenv("WANDB_API_KEY") else None

### Optional Google Colab use
Google Drive mounting is not required. In Colab, clone or upload the complete repository and run the notebook from the repository root. External model weights, when needed, must be placed under `external_materials/model_weights/`.


In [ ]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
!pip install transformers seaborn -q
!pip install umap-learn -q

import math
import umap.umap_ as umap
import os
import random
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, MaxNLocator
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report

import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
import string

# NLTK downloads (όπως στο αρχικό script)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

# Για αναπαραγωγιμότητα
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

LABEL_NAMES = ['Negative', 'Neutral', 'Positive']   # 0,1,2

In [ ]:
def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {
        "J": wordnet.ADJ,
        "N": wordnet.NOUN,
        "V": wordnet.VERB,
        "R": wordnet.ADV
    }
    return tag_dict.get(tag, wordnet.NOUN)


def clean_text(text, language="english"):
    stop_words = set(stopwords.words(language))
    lemmatizer = WordNetLemmatizer()

    if isinstance(text, str):
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        words = word_tokenize(text)
        filtered_words = [w for w in words if w not in stop_words]
        lemmatized_words = [lemmatizer.lemmatize(w, get_wordnet_pos(w)) for w in filtered_words]
        return " ".join(lemmatized_words)
    return text


def remove_stopwords_from_df(df, text_column="sentence", language="english"):
    df[text_column] = df[text_column].apply(lambda x: clean_text(x, language))
    return df


In [ ]:
# Φόρτωση αρχικού CSV
df = pd.read_csv('data/external/sentences_dataset_45269.csv')

"""print("\nΠροτάσεις πριν τον καθαρισμό!\n")
print(df.head(30))

# Καθαρισμός κειμένου (ίδιο με training)
df = remove_stopwords_from_df(df, text_column="sentence", language="english")

print("\nΠροτάσεις μετά τον καθαρισμό!\n")
print(df.head(30))"""


# Υπολογισμός αριθμού διπλών εγγραφών.
num_duplicates = df['sentence'].duplicated().sum()
print("Πλήθος διπλών εγγραφών στη στήλη sentence:", num_duplicates)

# Αποθήκευση του αρχικού πλήθους γραμμών.
before = len(df)

# Αφαίρεση διπλότυπων (κρατάμε την πρώτη εμφάνιση).
df = df.drop_duplicates(subset='sentence', keep='first')

# Υπολογισμός του πόσες γραμμές διαγράφηκαν.
after = len(df)
deleted = before - after
print("Πλήθος εγγραφών που διαγράφηκαν:", deleted)



# Train / temp split
train_data, temp_data = train_test_split(
    df,
    test_size=0.3,
    stratify=df['polarity'],
    random_state=SEED
)

# Val / Test split
val_data, test_data = train_test_split(
    temp_data,
    test_size=0.5,
    stratify=temp_data['polarity'],
    random_state=SEED
)

print("Train shape:", train_data.shape)
print("Val shape:", val_data.shape)
print("Test shape:", test_data.shape)

# Test set (θα το χρησιμοποιήσουμε για ΟΛΑ τα μοντέλα)
test_sentences = test_data['sentence'].tolist()
y_true = test_data['polarity'].tolist()   # 0 / 1 / 2

class_names = ['Negative', 'Neutral', 'Positive']  # προσαρμόζεται αν χρειάζεται
true_labels = [class_names[i] for i in y_true]



In [ ]:
# HEAT MAP KAI SCATTER PLOT ΓΙΑ ΤΑ ΜΟΝΤΕΛΑ ΤΗΣ ΟΙΚΟΓΕΝΕΙΑΣ BERT.

In [ ]:
BASE_DIR = "external_materials/model_weights"

def predict_for_model(folder_name, sentences, batch_size=16, max_length=256):
    model_dir = f"{BASE_DIR}/{folder_name}/saved_model"
    tok_dir   = f"{BASE_DIR}/{folder_name}/saved_tokenizer"

    print(f"\nΦόρτωση μοντέλου: {folder_name}")
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(tok_dir)
    model.eval()

    all_preds = []

    for i in range(0, len(sentences), batch_size):
        batch_texts = sentences[i:i+batch_size]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.no_grad():
            outputs = model(**enc)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()

        all_preds.extend(preds)

    return np.array(all_preds)


In [ ]:
def get_embeddings_for_model(folder_name, sentences, batch_size=16, max_length=256):
    model_dir = f"{BASE_DIR}/{folder_name}/saved_model"
    tok_dir   = f"{BASE_DIR}/{folder_name}/saved_tokenizer"

    print(f"\n[Embeddings] Φόρτωση μοντέλου: {folder_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_dir,
        output_hidden_states=True
    ).to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(tok_dir)
    model.eval()

    all_embs = []

    for i in range(0, len(sentences), batch_size):
        batch_texts = sentences[i:i+batch_size]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        with torch.no_grad():
            # Ζητάμε hidden states
            outputs = model(**enc, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]    # τελευταίο layer: (batch, seq, dim)
            cls_emb = hidden_states[:, 0, :]             # CLS token: (batch, dim)

        all_embs.append(cls_emb.cpu().numpy())

    embeddings = np.vstack(all_embs)   # (N, hidden_dim)
    return embeddings


In [ ]:
def save_classification_report_BERT_Family(model_name, y_true, y_pred, class_names, save_path):
    """
    Δημιουργεί PDF image με classification report για συγκεκριμένο μοντέλο.
    """

    # Παράγουμε report ως string
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=3
    )

    # Δημιουργούμε εικόνα
    fig, ax = plt.subplots(figsize=(8.5, 6))
    ax.axis("off")

    # Τοποθετούμε το κείμενο στο plot
    ax.text(
        0.01, 0.99,
        f"Classification Report — {model_name}\n\n{report}",
        fontsize=10,
        va="top",
        family="monospace"
    )

    fig.tight_layout()

    # Αποθήκευση PDF
    fig.savefig(save_path, format="pdf", bbox_inches="tight")
    plt.close(fig)

In [ ]:
print("Πληκτρολόγησε 1 για SciBERT, RoBERTa\n")
print("Πληκτρολόγησε 2 για ELECTRA, ALBERT, DistilBERT, BERT, DeBERTa, XLNet\n")

opt = int(input("Επιλογή: "))

if opt == 1:
    model_folders = ["SciBERT", "RoBERTa"] # "BERT", "DeBERTa", "XLNet",
else:
    model_folders = ["ELECTRA", "ALBERT", "DistilBERT", "BERT", "DeBERTa", "XLNet"]

#model_folders = ["BERT", "ELECTRA"]   # only 2 models

cms = {}  # dict: όνομα μοντέλου -> confusion matrix

for folder in model_folders:
    y_pred = predict_for_model(folder, test_sentences)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cms[folder] = cm

    report_path = f"outputs/figures/fig_comparable/{folder}_classification_report.pdf"
    save_classification_report_BERT_Family(folder, y_true, y_pred, LABEL_NAMES, report_path)
    print(f"Saved classification report for {folder} -> {report_path}")


# Τώρα: embeddings + UMAP
umap_results = {}   # folder_name -> 2D points (N, 2)

for folder in model_folders:
    emb = get_embeddings_for_model(folder, test_sentences, batch_size=16, max_length=256)
    print(f"{folder} embeddings shape:", emb.shape)

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        random_state=42
    )
    emb_2d = reducer.fit_transform(emb)   # (N, 2)

    umap_results[folder] = emb_2d

In [ ]:
sns.set(style="whitegrid", font_scale=1.0)

#model_folders = ["BERT", "ELECTRA"]
num_models = len(model_folders)

# 2 heatmaps ανά σειρά → άρα columns = 2
cols = 2
rows = math.ceil(num_models / cols)

fig, axes = plt.subplots(rows, cols, figsize=(9, 4 * rows))

# Κάνουμε flatten για εύκολη πρόσβαση
axes = axes.flatten()

for idx, folder in enumerate(model_folders):
    ax = axes[idx]
    cm = cms[folder]

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        cbar=False,
        ax=ax
    )
    ax.set_title(folder)
    ax.set_xlabel("Predicted Labels")
    ax.set_ylabel("True Labels")

# Απενεργοποίηση κενών subplots (αν έχεις μονό αριθμό μοντέλων)
for j in range(num_models, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
fig.savefig(
    'outputs/figures/fig_comparable/BERT_family_heatmaps.pdf',
    format='pdf',
    bbox_inches='tight',
    dpi=300
)


In [ ]:
sns.set(style="whitegrid", font_scale=1.0)

#num_models = len(model_folders)
cols = 2
rows = math.ceil(num_models / cols)

fig, axes = plt.subplots(rows, cols, figsize=(9, 4 * rows))
axes = axes.flatten()

for idx, folder in enumerate(model_folders):
    ax = axes[idx]
    emb_2d = umap_results[folder]

    df_plot = pd.DataFrame({
        "x": emb_2d[:, 0],
        "y": emb_2d[:, 1],
        "label": true_labels   # ground truth sentiment
    })

    palette = {
        "Negative": "tab:red",
        "Neutral": "tab:blue",
        "Positive": "tab:green"
    }

    sns.scatterplot(
        data=df_plot,
        x="x",
        y="y",
        hue="label",
        s=20,
        alpha=0.7,
        palette=palette,
        ax=ax
    )

    ax.set_title(f"{folder} Embeddings")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(title="Sentiment", loc="best", fontsize=10)

# Σβήνουμε κενά subplots αν περισσεύουν
for j in range(num_models, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
fig.savefig(
    'outputs/figures/fig_comparable/BERT_family_umap_scatter.pdf',
    format='pdf',
    bbox_inches='tight',
    dpi=300
)


Συνέχεια του χτισίματος του αρχείου Master Results File.
Το φορτώνω ξανά για δημιουργία των στηλών για τις προβλέψεις των encoders.

In [ ]:
# ---------------------------------------------------------------
# UPDATE MASTER RESULTS CSV WITH SciBERT & RoBERTa PREDICTIONS
# ---------------------------------------------------------------

MASTER_PATH = (
    "data/processed/Master_results.csv"
)

# Φόρτωση υπάρχοντος master αρχείου
master_df = pd.read_csv(MASTER_PATH)

print("Master CSV loaded. Shape:", master_df.shape)

# --------------------------------
# Υπολογισμός προβλέψεων
# --------------------------------
print("\nGenerating SciBERT predictions...")
scibert_preds = predict_for_model("SciBERT", test_sentences)

print("Generating RoBERTa predictions...")
roberta_preds = predict_for_model("RoBERTa", test_sentences)

# --------------------------------
# Έλεγχος συνέπειας
# --------------------------------
assert len(master_df) == len(scibert_preds) == len(roberta_preds), \
    " Mismatch in number of rows between Master CSV and predictions!"

# --------------------------------
# Προσθήκη στηλών
# --------------------------------
master_df["SciBERT_pred"] = scibert_preds
master_df["RoBERTa_pred"] = roberta_preds

# --------------------------------
# Αποθήκευση
# --------------------------------
master_df.to_csv(MASTER_PATH, index=False, encoding="utf-8")

print(f" Master_results.csv updated successfully -> {MASTER_PATH}")


In [ ]:
master_df

END